# Ablation recipient sampling and marker-pair support

**Role:** Analysis.

Inspect how recipient sampling affects center-marker/source-marker coverage, reuse and rare-fate support on a saved validation cohort. This is a sampling audit, independent of prediction outcomes or architecture. It contrasts uniform recipients and optional fate-stratified recipients; these estimate different populations and should not be pooled without explicit weights.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Select cohort and sampling rules

Restore a saved validation split without training. Sample recipient cells before computing any effects; retain the original full-graph node IDs.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'sampling_diagnostics'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CENTERS_PER_ORGANOID = 30  # Maximum sampled recipient centers per organoid.
STRATIFY_BY_CENTER = False  # Sample each center fate separately instead of uniformly.
HOPS = [1,2]  # Exact graph-distance rings to analyze.
SEED = 1701  # Random seed for reproducible sampling or initialization.

## Count available center–source pairs

Measure exact-hop pair availability and how often a physical source cell is reused. This distinguishes biological organoid support from a large number of overlapping perturbation cases.

In [ ]:
from src.graph.neighborhood import compute_hop_rings
rng=np.random.default_rng(SEED);rows=[]
for g in g_val:
    x=g.x.numpy()[:,:len(marker_names)]
    candidates=np.arange(len(x))
    if STRATIFY_BY_CENTER:
        centers=set()
        for j in range(len(marker_names)):
            ids=np.flatnonzero(x[:,j]>.5)
            centers.update(rng.choice(ids,min(CENTERS_PER_ORGANOID,len(ids)),replace=False).tolist())
    else:
        centers=rng.choice(candidates,min(CENTERS_PER_ORGANOID,len(x)),replace=False)
    for center in centers:
        rings=compute_hop_rings(g.edge_index,int(center),max(HOPS))
        center_labels=[marker_names[j] for j in np.flatnonzero(x[center]>.5)] or ['Unassigned']
        for hop in HOPS:
            for source in rings[hop]:
                source_labels=[marker_names[j] for j in np.flatnonzero(x[source]>.5)] or ['Unassigned']
                for recipient in center_labels:
                    for fate in source_labels:
                        rows.append(dict(organoid_str=g.organoid_str,center=int(center),source=int(source),hop=hop,recipient=recipient,fate=fate))
support=pd.DataFrame(rows)
summary=support.groupby(['hop','recipient','fate']).agg(organoids=('organoid_str','nunique'),candidate_pairs=('source','size'))
display(summary)
support.to_csv(OUTPUT_DIR/'sampling_support.csv.gz',index=False)
reuse=support.drop_duplicates(['organoid_str','center','source','hop']).groupby(['organoid_str','source','hop']).size()
reuse.hist(bins=30);plt.xlabel('Recipient uses per source cell');plt.ylabel('Physical source cells');plt.show()